# LAB | In God we trust, everyone else bring the data

**Scenario:** you are a marketing analyst. The CMO wants to split a **$500K monthly budget across the marketing channels** and asks you for a **statistically rigorous** recommendation. A false positive means moving money to a channel that isn't really better.

**How to use this notebook:**
- Cells marked **`# TODO`** are yours to complete. Replace every `...` with your code.
- 💡 **Hints** point to the statistics notebook (`03_statistical_analysis.ipynb`), where you saw each idea.
- Run the cells **in order**, top to bottom.

| Part | You will… | Concept from class |
|---|---|---|
| 1 | Find, load and clean a real dataset; compute metrics | A score is an **estimate** |
| 2 | Compare channels (t-test, Fisher) + effect size | p-value, **pick the test by the data**, Cohen's d |
| 3 | Correct for many comparisons | Bonferroni vs Benjamini-Hochberg |
| 4 | Check whether you have enough data | Power and sample size |
| 5 | Turn statistics into a recommendation | Confidence intervals, real vs worth it |

## Part 1 — Dataset discovery & exploration (45–60 min)
### Step 1: Find a dataset (max 30 min)

Search [Kaggle](https://www.kaggle.com/datasets) or [Hugging Face](https://huggingface.co/datasets) for: *marketing channels, advertising campaigns, campaign performance, ad spend, e-commerce conversion, A/B testing*.

Your dataset should have:
- **3+ groups** to compare (channels, campaigns, ad groups…)
- **performance columns**: impressions, clicks, conversions, cost, revenue (not always all of them)
- **repeated rows per group**, ideally one row per **day** or **week** per group (you need this for t-tests and power)

💡 **Hint:** a dataset with **one row per customer** (e.g. "responded yes/no") only works well for Fisher's test (Step 5). Prefer data with many rows per channel.

**✍️ Document your choice here:**
- Dataset name: SaaS Growth Pack (`channel_performance` table).
- Source + URL: [Hugging Face - SaaS Growth Pack](https://huggingface.co/datasets/solsticestudioai/saas-growth-pack). The data describe six fictional SaaS companies.
- Why I chose it: It has seven marketing channels and records impressions, clicks, conversions, cost (USD), and attributed revenue (`revenue_generated`, USD). I will filter to one company so that all seven channels can be compared over the same 90 days (January 1 to March 31, 2026). This gives 90 daily observations per channel before cleaning or excluding undefined metrics.
- What is one row? One marketing channel for one company on one calendar day. The full table contains 3,780 rows (90 days x 6 companies x 7 channels).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from pathlib import Path
from IPython.display import Markdown, display

from scipy import stats
from scipy.stats import fisher_exact, false_discovery_control   # false_discovery_control needs SciPy 1.11+

rng = np.random.default_rng(42)          # our "dice": fixed seed so results are repeatable
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)   # all charts / CSV / memo are saved here
sns.set_theme(style="whitegrid")
ALPHA = 0.05

### Step 2: Load, explore, clean

Rename your columns to these **standard names**, so the rest of the notebook works: `date, channel, impressions, clicks, conversions, cost, revenue`. If your dataset lacks a column, leave it out and skip the metrics that need it.

In [ ]:
# TODO: load your dataset
CSV_PATH = "your_dataset.csv"
COLUMN_MAP = {
    # "Your column name": "standard name",
    # "Date": "date", "Channel": "channel", "Spend": "cost", ...
}

raw = ...            # pd.read_csv(...) then .rename(columns=COLUMN_MAP)
# Hugging Face: from datasets import load_dataset; raw = load_dataset("...")["train"].to_pandas()

# TODO: print shape, column types and the first rows
...

In [ ]:
# TODO: clean the data
df = raw.copy()
num_cols = ["impressions", "clicks", "conversions", "cost", "revenue"]   # keep only the ones you have

# 1. Convert the date column with pd.to_datetime and numeric columns with pd.to_numeric(errors="coerce")
...
# 2. Check missing values (df.isna().sum()) and remove impossible rows
#    (negative values, clicks > impressions, conversions > clicks)
...
# 3. Print: number of groups, rows per group (value_counts), date range
...
# 4. Save the cleaned data
df.to_csv(OUT / "marketing_data.csv", index=False)

**✍️ Transformations I made:** …

💡 **Think:** how many rows does each channel have? That number is your **N** for the t-tests later.

### Step 3: Key marketing metrics (15–20 min)

| Metric | Formula |
|---|---|
| CTR | clicks / impressions |
| Conversion rate | conversions / clicks |
| CPA | cost / conversions (lower is better) |
| ROAS | revenue / cost (higher is better) |
| Profit · margin | revenue − cost · profit / revenue |

In [ ]:
# TODO: totals per channel, then the metrics
agg = df.groupby("channel")[num_cols].sum()
agg["CTR"] = ...
agg["conv_rate"] = ...
agg["CPA"] = ...
agg["ROAS"] = ...
agg["profit"] = ...
agg["profit_margin"] = ...
agg = agg.replace([np.inf, -np.inf], np.nan)
agg.round(3)

💡 **Hint: watch out for zero-cost channels** (e.g. SEO, organic). ROAS becomes infinite, and CPA becomes **$0**, which would make the channel look like the best one. Use `.where(agg.cost > 0)` to give them no CPA.

In [ ]:
# TODO: 2x3 grid of horizontal bar charts: CPA, ROAS, conversion rate, conversions, cost, profit
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
...
# Profit chart: add a vertical line at 0 (ax.axvline(0, color="black"))
plt.tight_layout()
plt.savefig(OUT / "group_metrics_overview.png", dpi=120)
plt.show()

In [ ]:
# TODO: per-day (or per-week) metrics -> these give the SPREAD you need for t-tests
df["CPA"] = ...
df["conv_rate"] = ...
df = df.replace([np.inf, -np.inf], np.nan)
paid = df[df.cost > 0]

# TODO: histograms (overlaid, alpha=0.5) and box plots (sns.boxplot) of daily CPA and conversion rate
...
plt.savefig(OUT / "group_distributions.png", dpi=120)
plt.show()

# TODO: spread per channel: mean, std and CV = std / mean of daily CPA
variability = ...

**✍️ Key insights:** best channel on each metric? Most variable channel? Outliers? Which channels look **close**?

💡 **Remember (§1–2 of the statistics notebook):** each channel's CPA is an **estimate** from a sample of days, not a fact.

## Part 2 — Pairwise comparisons (60–75 min)
### Step 4: t-tests + Cohen's d

For every **pair** of channels: is the gap in average daily CPA bigger than the day-to-day luck?

💡 **Hints:**
- `from itertools import combinations`: `combinations(groups, 2)` gives every pair once.
- Use **Welch's t-test**: `stats.ttest_ind(b, a, equal_var=False)`. Why Welch and not the default? Look at the test table in §4b.
- Cohen's d = (mean_b − mean_a) / √((var_a + var_b) / 2). Scale: < 0.2 negligible · < 0.5 small · < 0.8 medium · ≥ 0.8 large.

In [ ]:
def cohens_d(a, b):
    # TODO
    ...

def effect_label(d):
    # TODO: return "negligible" / "small" / "medium" / "large"
    ...

def pairwise_ttests(data, metric):
    clean = data[["channel", metric]].replace([np.inf, -np.inf], np.nan).dropna()
    clean = clean[clean[metric] > 0]
    rows = []
    for a, b in combinations(sorted(clean.channel.unique()), 2):
        xa = ...
        xb = ...
        t, p = ...
        diff = ...
        d = ...
        rows.append({"group_a": a, "group_b": b, "mean_a": xa.mean(), "mean_b": xb.mean(),
                     "diff": diff, "pct_diff": ..., "t_stat": t, "p_value": p,
                     "cohens_d": d, "effect": effect_label(d), "significant": p < ALPHA})
    res = pd.DataFrame(rows)
    print(f"{metric}: {len(res)} comparisons, {res.significant.sum()} significant")
    return res

cpa_res = pairwise_ttests(paid, "CPA")
cvr_t_res = pairwise_ttests(df, "conv_rate")

In [ ]:
# TODO: heatmap of CPA p-values (channels x channels)
# 1. Square DataFrame filled with 1.0 (diagonal = channel vs itself)
# 2. Fill both [a, b] and [b, a] with the p-value
# 3. sns.heatmap(..., annot=True, cmap="RdYlGn_r", cbar_kws={"label": "p-value"})
...
plt.savefig(OUT / "metric_comparison_heatmap.png", dpi=120)
plt.show()

**✍️ Which channels differ significantly? Which pairs can't be told apart?**

💡 **Remember (§3):** a big p-value means **"can't tell"**, not "no difference".

### Step 5: Fisher's exact test on conversions (30–35 min)

Each click either converts or doesn't: **right/wrong data**, so Fisher's exact test (§4b).

```
[[conversions_A, non_conversions_A],
 [conversions_B, non_conversions_B]]
```

In [ ]:
# TODO: totals per channel
conv = df.groupby("channel")[["conversions", "clicks"]].sum()
conv["non_conversions"] = ...
conv["conv_rate"] = ...

rows = []
for a, b in combinations(sorted(conv.index), 2):
    table = ...
    odds_ratio, p = fisher_exact(table, alternative="two-sided")
    # TODO: rates, difference in percentage points, % difference
    rows.append({...})
fisher_res = pd.DataFrame(rows)
...

In [ ]:
# TODO: horizontal bar chart of conversion rates, sorted, with % labels (plt.bar_label)
...
plt.savefig(OUT / "rate_comparison.png", dpi=120)
plt.show()

# TODO: compare Fisher with the t-test on daily conversion rates (cvr_t_res).
# How many pairs are significant in one test but not the other?
...

**✍️ Answer:**
1. Which channels have significantly different conversion rates?
2. Are the differences **large enough to matter**? (Look at the gap in percentage points.)
3. Fisher vs t-test: do they agree? If not, **why**?

💡 **Hint for 3:** compare the **N** of each test. For Fisher, N = number of clicks. For the t-test, N = number of days. What did §3b show about large N and small gaps?

## Part 3 — Multiple comparisons correction (45–60 min)
### Step 6: Bonferroni and Benjamini-Hochberg

💡 **Hints (§8 of the statistics notebook):**
- Bonferroni: significant if `p < ALPHA / number_of_tests`
- BH: `false_discovery_control(p_values, method="bh")` returns **adjusted** p-values; significant if adjusted < ALPHA

In [ ]:
families = {"CPA (t-test)": cpa_res, "Conv. rate (t-test)": cvr_t_res, "Conv. rate (Fisher)": fisher_res}

# TODO: total number of tests and expected false positives (tests x 0.05)
...

rows = []
for name, res in families.items():
    m = len(res)
    res["significant_bonferroni"] = ...
    res["p_value_fdr"] = ...
    res["significant_fdr"] = ...
    rows.append({"family": name, "tests": m, "uncorrected": ..., "Bonferroni": ..., "BH-FDR": ...})
correction_summary = pd.DataFrame(rows).set_index("family")
correction_summary

In [ ]:
# TODO: which comparisons lose significance after each correction? Print them.
...

# TODO: grouped bar chart: uncorrected vs Bonferroni vs BH per family
...
plt.savefig(OUT / "correction_comparison.png", dpi=120)
plt.show()

**✍️ Answer:**
1. How many "significant" results disappeared after correction?
2. Which correction fits a **budget decision**, and why? (Slide: *choose by the cost of a false winner*.)
3. What does this mean for moving budget?

## Part 4 — Power analysis (30–45 min)
### Step 7: Do we have enough data?

**Power** = the chance of catching a difference that is **really** there (§5). Simulate many experiments where you **know** the truth:
- channel A: daily CPA ~ normal(mean = `base_cpa`, std = 15% of `base_cpa`)
- channel B: the same, but mean = `base_cpa × (1 + true_diff_pct)`
- cap low values at 50% of `base_cpa` (`np.maximum(x, 0.5 * base_cpa)`)
- t-test → count how often p < 0.05

💡 **Speed hint:** draw all simulations at once with shape `(n_sim, n_days)` and use `stats.ttest_ind(a, b, axis=1)`. A plain loop also works, but use `n_sim=300` then.

In [ ]:
def empirical_power_cpa(true_diff_pct, base_cpa, n_days, n_sim=1000, alpha=0.05):
    # TODO
    ...

BASE_CPA = ...                       # e.g. median CPA of the paid channels
effects = [0.05, 0.10, 0.15, 0.20]
days = [30, 60, 90, 120, 180]

# TODO: DataFrame with columns effect_size_pct, n_days, power
power_df = ...

# TODO: one line per effect size, red dashed line at 0.8
...
plt.savefig(OUT / "power_analysis_cpa.png", dpi=120)
plt.show()

In [ ]:
# TODO: minimum days for 80% power, per effect size. Is our data enough? ("sufficient" / "insufficient")
...

# TODO: for each CPA pair significant after BH: observed % difference (smaller mean = baseline),
#       power with your number of days, and days needed for 80% if power < 0.8
...

**✍️ Answer:**
1. What size of CPA difference can your data detect reliably?
2. How many days would you need to detect a 5% difference?
3. Look at your non-significant pairs: "no difference" or "not enough data"?

💡 **Remember:** half the difference → about **4× the data**.

## Part 5 — Business recommendations (30–45 min)
### Step 8: From statistics to a decision

In [ ]:
# TODO: summary of significant findings (use the BH-corrected results)
#  - CPA: which channel is cheaper, by how much ($), adjusted p-value, Cohen's d
#  - Conversion rate: which channel is higher, by how many points, adjusted p-value
...

In [ ]:
def bootstrap_ci(data, n_bootstrap=1000, ci_level=0.95):
    # TODO: resample with replacement, take the mean each time, return the percentiles
    ...

# TODO: table: channel, mean daily CPA, 95% CI low, 95% CI high
...

In [ ]:
# TODO: composite score (paid channels only)
#  - rank CPA so the LOWEST CPA gets the highest rank; rank ROAS so the HIGHEST ROAS gets the highest rank
#  - composite = 0.5 * CPA_rank + 0.5 * ROAS_rank
...

# TODO: split $500,000 in proportion to the composite score, with min 5% and max 35% per channel
...

💡 **Before you write the memo, think:**
- Two channels whose CPA difference is **not significant** still get different ranks. Is that fair?
- The top channel may be **small**. Can it really absorb a big budget at the same CPA?
- Which "significant" differences are **too small to matter**?

In [ ]:
# TODO: write the executive memo as Markdown and save it
memo = f"""# Executive memo — Marketing budget allocation

**To:** CMO · **From:** ... · **Date:** ... · **Dataset:** ... · **Period:** ...

## Executive summary
...

## Key findings
- Top performers: ...
- Statistically significant differences (after correction): ...
- Data adequacy (power): ...

## Recommendations
...

## Statistical caveats
- Dataset source and limits: ...
- Multiple comparisons correction: ...
- Confidence intervals: ...
- External factors: ...
- Statistical vs practical significance: ...
- Power limitations: ...

## Next steps
...
"""
(OUT / "executive_memo.md").write_text(memo, encoding="utf-8")
display(Markdown(memo))

## Reflection (optional, recommended): `reflection.md`
1. What surprised you about the results?
2. How did multiple-comparison correction change your conclusions?
3. What are the limits of this analysis?
4. How would you explain the findings to a non-technical CMO in one minute?

## Submission checklist
- [ ] GitHub repo with only this lab, plus a `README.md` (how to run + file map)
- [ ] `lab_proof.md` as the entry point (code/workflow, one run with its input, the evidence, the final output, one failure or limitation you hit)
- [ ] Notebook(s) run top to bottom without errors
- [ ] `outputs/`: `marketing_data.csv`, the 5 charts, `executive_memo.md`